# AxioGo Lakehouse - Gold Layer (Insight) Comprehensive Schema

## Schema Pattern: **STAR SCHEMA**

---

## DIMENSION TABLES (7)

### 1. dim_vehicle
**PK:** vehicle_id | **Source:** forge.vehicle_master
```sql
vehicle_id, registration_number, vin, manufacturer, model, variant,
manufacturing_year, purchase_date, vehicle_class, fuel_type, engine_type, 
engine_capacity_cc, transmission, drivetrain, color, seating_capacity,
gross_vehicle_weight_kg, payload_capacity_kg, fuel_tank_capacity_l,
warranty_expiry, insurance_policy_id, current_odometer_km, assigned_depot,
status, last_service_date, vehicle_age_years, days_since_purchase,
is_under_warranty, needs_service, is_operational
```

### 2. dim_driver  
**PK:** driver_id | **Source:** forge.driver_master
```sql
driver_id, driver_name, license_number, license_type, experience_years,
risk_profile, driving_style, assigned_vehicle_id, joining_date, home_depot,
days_with_company, years_with_company, experience_category,
is_high_risk, is_experienced, is_new_driver
```

### 3. dim_route
**PK:** route_id | **Source:** forge.route_master
```sql
route_id, source, destination, distance_km, estimated_duration_min,
road_type, traffic_level, expected_avg_speed_kmph,
is_valid_distance, is_valid_duration
```

### 4. dim_weather
**PK:** weather_id | **Source:** forge.weather
```sql
weather_id, location, temperature, humidity, wind_speed, visibility,
weather_condition, precipitation, is_adverse_weather
```

### 5. dim_date
**PK:** date_key | **Source:** Generated
```sql
date_key, full_date, year, quarter, quarter_name, month, month_name,
week_of_year, day_of_month, day_of_week, day_name,
is_weekend, is_holiday, fiscal_year, fiscal_quarter
```

### 6. dim_time
**PK:** time_key | **Source:** Generated  
```sql
time_key, hour, minute, time_of_day (Morning/Afternoon/Evening/Night),
shift (Day/Night), is_peak_hour, is_business_hour
```

### 7. dim_location
**PK:** location_key | **Source:** Generated from routes/GPS
```sql
location_key, city, region, depot, latitude, longitude, zone
```

---

## FACT TABLES (10 - Comprehensive)

### TRANSACTION FACTS

### 1. fact_trip_detail
**PK:** trip_id | **Grain:** One row per valid completed trip  
**Source:** forge.trip_master + fuel_transactions + aggregated telemetry
```sql
-- Keys
trip_id (PK), vehicle_id (FK), driver_id (FK), route_id (FK), weather_id (FK),
trip_date_key (FK), start_time_key (FK), end_time_key (FK),

-- Trip Basics
start_time, end_time, trip_status, trip_date, trip_hour,

-- Distance & Duration
distance_km, trip_duration_minutes, average_speed_kmph,
calculated_avg_speed, is_speed_mismatch,

-- Fuel Metrics (joined from fuel_transactions)
fuel_quantity_l, fuel_cost, fuel_price_per_l, fuel_station,
fuel_efficiency_kmpl, fuel_cost_per_km, is_large_refuel,

-- Performance vs Route
route_distance_km, route_adherence_pct,
expected_avg_speed_kmph, speed_variance_pct,

-- Validation Flags
is_completed, is_cancelled, is_valid_duration, is_valid_trip,
is_on_time, is_adverse_weather
```

### 2. fact_engine_telemetry
**PK:** engine_record_id | **Grain:** One row per engine telemetry reading  
**Source:** forge.core_engine
```sql
-- Keys
engine_record_id (PK), vehicle_id (FK), trip_id (FK), 
record_date_key (FK), record_time_key (FK), timestamp,

-- Engine Identifiers
engine_serial_no, engine_model,

-- Core Metrics
engine_rpm, engine_load, engine_status, engine_health_score,

-- Temperature Monitoring
coolant_temperature, oil_temperature, exhaust_temperature,
is_overheating, is_valid_coolant_temp, is_valid_oil_temp,

-- Pressure Monitoring
coolant_pressure, oil_pressure, fuel_pressure, turbo_boost,

-- Electrical
battery_voltage,

-- Fuel System
fuel_injection_rate,

-- Vibration & Sensors
engine_vibration, knock_sensor,

-- Diagnostics
fault_code, warning_level, has_fault, is_critical_warning,

-- Performance Indicators
is_healthy, is_high_load, engine_performance,
custom_health_score, health_score_mismatch, has_health_score_issue
```

### 3. fact_driver_behavior
**PK:** behaviour_id | **Grain:** One row per driver behavior telemetry reading  
**Source:** forge.driver_behavior
```sql
-- Keys
behaviour_id (PK), vehicle_id (FK), driver_id (FK), trip_id (FK),
record_date_key (FK), record_time_key (FK), timestamp,

-- Speed Metrics
average_speed, maximum_speed, overspeed_events,

-- Harsh Events
harsh_braking_count, rapid_acceleration_count, hard_cornering_count,
lane_departure_events,

-- Idle & Efficiency
idle_time, eco_driving_score,

-- Driving Hours
continuous_driving_hours, night_driving_hours,
is_excessive_hours, is_night_driver,

-- Safety Scores
driver_fatigue_score, driver_distraction_score,
is_fatigued, is_distracted,

-- Safety Compliance
seatbelt_status, phone_usage, is_unsafe_conditions,

-- Risk Assessment
composite_risk_score, is_high_risk_driving
```

### 4. fact_maintenance
**PK:** service_id | **Grain:** One row per maintenance event  
**Source:** forge.maintenance
```sql
-- Keys
service_id (PK), vehicle_id (FK), service_date_key (FK), 
workshop_location_key (FK),

-- Service Details
engine_serial_no, service_date, service_type, service_category,
service_status, service_year, service_month, service_hour,

-- Diagnosis
fault_code, diagnosis, has_fault_code, fault_code_count,

-- Service Provider
technician, workshop,

-- Work Performed
parts_replaced,

-- Costs
labour_cost, parts_cost, total_cost, calculated_total_cost,
is_cost_mismatch, cost_difference, cost_per_hour,
is_expensive_repair,

-- Downtime
downtime_hours, downtime_days, is_high_downtime,

-- Next Service
next_service_due, days_until_next_service, is_overdue,

-- Warranty
warranty_claim, is_warranty_claim,

-- Service Type Flags
is_preventive, is_breakdown
```

### 5. fact_insurance_claim
**PK:** claim_id | **Grain:** One row per insurance claim  
**Source:** forge.insurance_claims
```sql
-- Keys
claim_id (PK), vehicle_id (FK), driver_id (FK), trip_id (FK),
accident_date_key (FK), accident_location_key (FK), accident_timestamp,

-- Location & Conditions
accident_location, weather_condition, road_condition,
is_adverse_weather, is_poor_road,

-- Accident Details
collision_type, damaged_component, severity, is_severe,

-- Financial
estimated_repair_cost, claim_amount, cost_difference,
is_high_value_claim, is_claim_over_estimate,

-- Fraud Detection
fraud_flag, is_potential_fraud,

-- Status
claim_status
```

### 6. fact_fuel_transaction
**PK:** fuel_id | **Grain:** One row per fuel purchase  
**Source:** forge.fuel_transactions
```sql
-- Keys
fuel_id (PK), vehicle_id (FK), trip_id (FK), 
transaction_date_key (FK), fuel_station_location_key (FK), 
timestamp, transaction_date,

-- Station & Type
fuel_station, fuel_type, payment_method,

-- Quantities & Pricing
fuel_quantity_l, fuel_price_per_l, total_cost,
calculated_total_cost, is_cost_mismatch,

-- Validation
is_valid_quantity, is_valid_price, is_large_refuel, is_complete
```

### AGGREGATED FACTS

### 7. fact_trip_daily_summary
**PK:** vehicle_id + driver_id + date_key | **Grain:** Daily aggregates per vehicle-driver  
**Source:** Aggregated from fact_trip_detail
```sql
-- Keys
vehicle_id (FK), driver_id (FK), date_key (FK), trip_date,

-- Trip Counts
total_trips, completed_trips, cancelled_trips,

-- Distance & Duration
total_distance_km, avg_distance_per_trip, total_duration_minutes,

-- Fuel
total_fuel_quantity_l, total_fuel_cost, avg_fuel_efficiency_kmpl,

-- Performance
avg_speed_kmph, max_speed_kmph,

-- Risk Events
total_harsh_braking, total_rapid_acceleration, total_overspeed_events,
avg_risk_score,

-- Financial
total_trip_revenue, total_trip_cost, gross_profit
```

### 8. fact_vehicle_monthly
**PK:** vehicle_id + year_month | **Grain:** Monthly vehicle performance  
**Source:** Aggregated from multiple fact tables
```sql
-- Keys & Time
vehicle_id (FK), year_month, year, month,

-- Trip Metrics
total_trips, completed_trips, cancelled_trips, trip_completion_rate,
total_distance_km, avg_trip_distance_km, total_trip_hours,

-- Utilization
active_days, utilization_rate, trips_per_active_day, idle_days,

-- Fuel Performance
total_fuel_quantity_l, total_fuel_cost, avg_fuel_efficiency_kmpl,
fuel_cost_per_km,

-- Maintenance
maintenance_event_count, preventive_count, breakdown_count,
total_maintenance_cost, total_downtime_hours,

-- Insurance
claim_count, total_claim_amount, severe_claim_count,

-- Engine Health (from telemetry)
avg_engine_health_score, min_engine_health_score,
critical_warning_count, fault_code_count,

-- Financial
total_revenue, total_cost, gross_profit, profit_margin,
cost_per_km, revenue_per_km,

-- Performance Flags
is_high_performer, is_underutilized, needs_attention, has_overdue_service
```

### 9. fact_driver_monthly
**PK:** driver_id + year_month | **Grain:** Monthly driver performance  
**Source:** Aggregated from fact_trip_detail + fact_driver_behavior
```sql
-- Keys & Time
driver_id (FK), year_month, year, month,

-- Trip Metrics
total_trips, completed_trips, total_distance_km, total_driving_hours,
active_days, trips_per_day, avg_trip_distance_km,

-- Time Patterns
night_driving_hours, night_trip_count, night_trip_pct,

-- Safety & Behavior
avg_composite_risk_score, max_risk_score,
total_harsh_braking, total_rapid_acceleration, total_hard_cornering,
total_overspeed_events, total_lane_departure,
avg_harsh_events_per_trip,

-- Fatigue & Violations
avg_fatigue_score, avg_distraction_score,
max_continuous_driving_hours, excessive_hours_count,
phone_usage_violations, seatbelt_violations,

-- Eco Driving
avg_eco_driving_score, avg_fuel_efficiency_kmpl, total_idle_time_hours,

-- Insurance
claim_count, at_fault_claim_count, total_claim_amount, severe_claim_count,

-- Speed
avg_speed_kmph, max_speed_kmph, overspeed_trip_count, overspeed_trip_pct,

-- Performance Ratings
risk_category, safety_rating, performance_rating,
is_high_risk_driver, needs_training, is_top_performer, requires_intervention
```

### 10. fact_fleet_daily_kpi
**PK:** kpi_date | **Grain:** Daily fleet-wide KPIs  
**Source:** Aggregated from all fact tables
```sql
-- Keys
kpi_date_key (FK), kpi_date,

-- Fleet Size
total_vehicles, active_vehicles, vehicles_in_maintenance,
total_drivers, active_drivers,

-- Operations
total_trips, completed_trips, cancelled_trips, trip_completion_rate,
total_distance_km, avg_distance_per_trip,

-- Utilization
fleet_utilization_rate, driver_utilization_rate,
avg_trips_per_vehicle, avg_trips_per_driver,

-- Fuel
total_fuel_quantity_l, total_fuel_cost, avg_fuel_efficiency_kmpl,
fuel_cost_per_km,

-- Maintenance
maintenance_events, total_maintenance_cost, vehicles_in_service,
total_downtime_hours,

-- Safety
claim_count, severe_claim_count, total_claim_amount,
total_harsh_events, high_risk_trip_count,

-- Engine Health
avg_engine_health_score, critical_warning_count, vehicles_with_faults,

-- Financial
total_revenue, total_operating_cost, gross_profit, profit_margin,
cost_per_km, revenue_per_km,

-- Performance vs Target
trips_vs_target_pct, revenue_vs_target_pct, cost_vs_target_pct
```

# Comprehensive Star Schema - Entity Relationship Diagram

```
                         DIMENSION LAYER
                         
┌─────────────┐  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐
│ dim_vehicle │  │ dim_driver  │  │ dim_route   │  │ dim_weather │
│ PK:vehicle  │  │ PK:driver   │  │ PK:route    │  │ PK:weather  │
└──────┬──────┘  └──────┬──────┘  └──────┬──────┘  └──────┬──────┘
       │                │                │                │
       │  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐
       │  │  dim_date   │  │  dim_time   │  │dim_location │
       │  │ PK:date_key │  │ PK:time_key │  │ PK:location │
       │  └──────┬──────┘  └──────┬──────┘  └──────────────┘
       │         │                │
       │         │                │
       └─────────┼────────────────┼────────────────┐
                 │                │                │
═════════════════════════════════════════════════════════════════
                    TRANSACTION FACT LAYER
═════════════════════════════════════════════════════════════════
                 │                │                │
        ┌────────▼────────────────▼────────────────▼────────┐
        │         fact_trip_detail                          │
        │  PK: trip_id                                      │
        │  FK: vehicle, driver, route, weather, date, time  │
        │  Metrics: distance, duration, fuel, performance   │
        └───────────────────────┬───────────────────────────┘
                                │
                ┌───────────────┼───────────────┐
                │               │               │
       ┌────────▼────────┐  ┌──▼──────────┐  ┌▼──────────────┐
       │fact_engine_     │  │fact_driver_ │  │fact_fuel_     │
       │telemetry        │  │behavior     │  │transaction    │
       │PK:engine_record │  │PK:behaviour │  │PK:fuel_id     │
       │FK:vehicle,trip  │  │FK:veh,drv,  │  │FK:vehicle,    │
       │                 │  │   trip      │  │   trip        │
       │All engine       │  │All behavior │  │All fuel       │
       │metrics          │  │metrics      │  │purchases      │
       └─────────────────┘  └─────────────┘  └───────────────┘

       ┌─────────────────┐  ┌──────────────────┐
       │fact_maintenance │  │fact_insurance_   │
       │PK:service_id    │  │claim             │
       │FK:vehicle,date  │  │PK:claim_id       │
       │                 │  │FK:vehicle,driver,│
       │All maintenance  │  │   trip,date      │
       │details          │  │All claim details │
       └─────────────────┘  └──────────────────┘

═════════════════════════════════════════════════════════════════
                    AGGREGATED FACT LAYER
═════════════════════════════════════════════════════════════════

       ┌────────────────────┐  ┌────────────────────┐
       │fact_trip_daily_    │  │fact_vehicle_       │
       │summary             │  │monthly             │
       │PK:veh+drv+date     │  │PK:vehicle+month    │
       │Daily aggregates    │  │Monthly aggregates  │
       └────────────────────┘  └────────────────────┘

       ┌────────────────────┐  ┌────────────────────┐
       │fact_driver_        │  │fact_fleet_daily_   │
       │monthly             │  │kpi                 │
       │PK:driver+month     │  │PK:kpi_date         │
       │Monthly aggregates  │  │Fleet-wide daily    │
       └────────────────────┘  └────────────────────┘
```

---

## Data Flow

```
Forge Layer (Silver)  →  Gold Layer (Insight)

1. trip_master + fuel_transactions → fact_trip_detail
2. core_engine → fact_engine_telemetry
3. driver_behavior → fact_driver_behavior
4. maintenance → fact_maintenance
5. insurance_claims → fact_insurance_claim
6. fuel_transactions → fact_fuel_transaction

Aggregation:
fact_trip_detail → fact_trip_daily_summary
fact_trip_detail + fact_maintenance + fact_insurance_claim → fact_vehicle_monthly
fact_trip_detail + fact_driver_behavior → fact_driver_monthly
All facts → fact_fleet_daily_kpi
```

In [0]:
# Comprehensive Gold Layer Schema Summary

summary = """
================================================================================
GOLD LAYER (INSIGHT) - COMPREHENSIVE SCHEMA
Schema: workspace.insight
Pattern: STAR SCHEMA
================================================================================

DIMENSION TABLES (7)
--------------------
1. dim_vehicle      - PK: vehicle_id       (500 vehicles)
2. dim_driver       - PK: driver_id        (700 drivers)
3. dim_route        - PK: route_id         (100 routes)
4. dim_weather      - PK: weather_id       (weather conditions)
5. dim_date         - PK: date_key         (calendar)
6. dim_time         - PK: time_key         (time of day)
7. dim_location     - PK: location_key     (geographic)


TRANSACTION FACT TABLES (6) - Detailed granular data
----------------------------------------------------
1. fact_trip_detail
   - Grain: One row per valid completed trip (~7,460)
   - Includes: distance, duration, fuel, performance vs route
   - FKs: vehicle, driver, route, weather, date, time

2. fact_engine_telemetry
   - Grain: One row per engine telemetry reading (5,000+)
   - Includes: RPM, load, temps, pressures, health scores, fault codes
   - FKs: vehicle, trip, date, time

3. fact_driver_behavior
   - Grain: One row per driver behavior reading (5,000+)
   - Includes: speed, harsh events, fatigue, distraction, risk scores
   - FKs: vehicle, driver, trip, date, time

4. fact_maintenance
   - Grain: One row per maintenance event (4,000)
   - Includes: service type, costs, downtime, fault codes, warranty
   - FKs: vehicle, date

5. fact_insurance_claim
   - Grain: One row per insurance claim (2,000)
   - Includes: accident details, costs, severity, fraud flags
   - FKs: vehicle, driver, trip, date

6. fact_fuel_transaction
   - Grain: One row per fuel purchase (8,000)
   - Includes: quantity, cost, station, fuel type
   - FKs: vehicle, trip, date


AGGREGATED FACT TABLES (4) - Pre-computed summaries
----------------------------------------------------
7. fact_trip_daily_summary
   - Grain: Daily per vehicle-driver combination
   - Includes: trip counts, totals, averages, risk metrics

8. fact_vehicle_monthly
   - Grain: Monthly per vehicle
   - Includes: trips, utilization, fuel, maintenance, claims, health, financials

9. fact_driver_monthly
   - Grain: Monthly per driver
   - Includes: trips, safety, behavior, claims, performance ratings

10. fact_fleet_daily_kpi
    - Grain: Daily fleet-wide
    - Includes: fleet metrics, operations, safety, financials, vs target


KEY FEATURES
------------
✓ All engine telemetry preserved (temps, pressures, health, faults)
✓ All driver behavior preserved (harsh events, fatigue, distraction)
✓ All maintenance details preserved (costs, downtime, fault codes)
✓ All claim details preserved (severity, fraud, conditions)
✓ All fuel transactions preserved
✓ Weather context linked to trips
✓ Time-of-day analysis enabled
✓ Pre-aggregated tables for fast dashboards


CRITICAL RULES
--------------
⚠️  fact_trip_detail: ONLY valid completed trips (is_valid_trip = true)
⚠️  fact_engine_telemetry: Full granular engine monitoring data
⚠️  fact_driver_behavior: Full granular driver behavior data
⚠️  All dimensions: SCD Type 1 (current state only)
⚠️  Aggregated facts: Derived from transaction facts


USE CASES ENABLED
-----------------
• Trip-level analysis with full context
• Real-time engine health monitoring
• Driver safety & behavior analysis
• Maintenance cost & downtime tracking
• Insurance claim analysis & fraud detection
• Fuel efficiency & cost optimization
• Fleet utilization & performance
• Executive dashboards & KPIs
• Predictive maintenance (engine patterns)
• Driver training needs identification
• Route profitability analysis
• Weather impact on operations

================================================================================
"""

print(summary)

In [0]:
%sql
-- Create Gold Layer Schema
CREATE SCHEMA IF NOT EXISTS workspace.insight
COMMENT 'Gold Layer - Star Schema for Analytics and Reporting';

In [0]:
%sql
-- Dimension: Vehicle Master (reviewed)
CREATE OR REPLACE TABLE workspace.insight.dim_vehicle AS
SELECT 
  vehicle_id,
  registration_number,
  vin,
  manufacturer,
  model,
  variant,
  manufacturing_year,
  purchase_date,
  vehicle_class,
  fuel_type,
  engine_type,
  engine_capacity_cc,
  transmission,
  drivetrain,
  color,
  seating_capacity,
  gross_vehicle_weight_kg,
  payload_capacity_kg,
  fuel_tank_capacity_l,
  warranty_expiry,
  insurance_policy_id,
  current_odometer_km,
  assigned_depot,
  status,
  last_service_date,
  vehicle_age_years,
  days_since_purchase,
  is_under_warranty,
  needs_service,
  is_operational
FROM workspace.forge.vehicle_master;

In [0]:
%sql
-- Dimension: Driver Master
CREATE OR REPLACE TABLE workspace.insight.dim_driver AS
SELECT 
  driver_id,
  driver_name,
  license_number,
  license_type,
  experience_years,
  risk_profile,
  driving_style,
  assigned_vehicle_id,
  joining_date,
  home_depot,
  days_with_company,
  years_with_company,
  experience_category,
  is_high_risk,
  is_experienced,
  is_new_driver
FROM workspace.forge.driver_master;

In [0]:
%sql
-- Dimension: Route Master
CREATE OR REPLACE TABLE workspace.insight.dim_route AS
SELECT 
  route_id,
  source,
  destination,
  distance_km,
  estimated_duration_min,
  road_type,
  traffic_level,
  expected_avg_speed_kmph,
  is_valid_distance,
  is_valid_duration
FROM workspace.forge.route_master;

In [0]:
%sql
-- Dimension: Weather
CREATE OR REPLACE TABLE workspace.insight.dim_weather AS
SELECT 
  weather_id,
  location,
  temperature,
  humidity,
  wind_speed,
  visibility,
  weather_condition,
  precipitation,
  is_adverse_weather
FROM workspace.forge.weather;

In [0]:
%sql
-- Dimension: Date
CREATE OR REPLACE TABLE workspace.insight.dim_date AS
WITH date_range AS (
  SELECT explode(sequence(
    to_date('2024-01-01'), 
    to_date('2027-12-31'), 
    interval 1 day
  )) AS full_date
)
SELECT 
  CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
  full_date,
  year(full_date) AS year,
  quarter(full_date) AS quarter,
  CONCAT('Q', quarter(full_date)) AS quarter_name,
  month(full_date) AS month,
  date_format(full_date, 'MMMM') AS month_name,
  weekofyear(full_date) AS week_of_year,
  dayofmonth(full_date) AS day_of_month,
  dayofweek(full_date) AS day_of_week,
  date_format(full_date, 'EEEE') AS day_name,
  CASE WHEN dayofweek(full_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  FALSE AS is_holiday,
  year(full_date) AS fiscal_year,
  quarter(full_date) AS fiscal_quarter
FROM date_range;

In [0]:
%sql
-- Dimension: Time
CREATE OR REPLACE TABLE workspace.insight.dim_time AS
WITH time_range AS (
  SELECT explode(sequence(0, 1439)) AS minute_of_day
)
SELECT 
  minute_of_day AS time_key,
  CAST(minute_of_day / 60 AS INT) AS hour,
  minute_of_day % 60 AS minute,
  CASE 
    WHEN CAST(minute_of_day / 60 AS INT) BETWEEN 6 AND 11 THEN 'Morning'
    WHEN CAST(minute_of_day / 60 AS INT) BETWEEN 12 AND 17 THEN 'Afternoon'
    WHEN CAST(minute_of_day / 60 AS INT) BETWEEN 18 AND 21 THEN 'Evening'
    ELSE 'Night'
  END AS time_of_day,
  CASE 
    WHEN CAST(minute_of_day / 60 AS INT) BETWEEN 6 AND 18 THEN 'Day Shift'
    ELSE 'Night Shift'
  END AS shift,
  CASE 
    WHEN CAST(minute_of_day / 60 AS INT) IN (7, 8, 9, 17, 18, 19) THEN TRUE 
    ELSE FALSE 
  END AS is_peak_hour,
  CASE 
    WHEN CAST(minute_of_day / 60 AS INT) BETWEEN 9 AND 17 THEN TRUE 
    ELSE FALSE 
  END AS is_business_hour
FROM time_range;

In [0]:
%sql
-- Dimension: Location (from routes)
CREATE OR REPLACE TABLE workspace.insight.dim_location AS
WITH locations AS (
  SELECT DISTINCT source AS location FROM workspace.forge.route_master
  UNION
  SELECT DISTINCT destination AS location FROM workspace.forge.route_master
  UNION
  SELECT DISTINCT assigned_depot AS location FROM workspace.forge.vehicle_master
  WHERE assigned_depot IS NOT NULL
),
location_with_keys AS (
  SELECT 
    ROW_NUMBER() OVER (ORDER BY location) AS location_key,
    location,
    location AS city,
    CASE 
      WHEN location IN ('Mumbai', 'Pune', 'Nagpur') THEN 'Maharashtra'
      WHEN location IN ('Delhi', 'Gurgaon', 'Noida') THEN 'NCR'
      WHEN location IN ('Bangalore', 'Mysore') THEN 'Karnataka'
      ELSE 'Other'
    END AS region,
    location AS depot,
    NULL AS latitude,
    NULL AS longitude,
    'Zone-1' AS zone
  FROM locations
)
SELECT * FROM location_with_keys;

In [0]:
%sql
-- Fact: Trip Detail (with fuel joined)
CREATE OR REPLACE TABLE workspace.insight.fact_trip_detail AS
SELECT 
  -- Keys
  t.trip_id,
  t.vehicle_id,
  t.driver_id,
  t.route_id,
  w.weather_id,
  CAST(date_format(t.start_time, 'yyyyMMdd') AS INT) AS trip_date_key,
  hour(t.start_time) * 60 + minute(t.start_time) AS start_time_key,
  hour(t.end_time) * 60 + minute(t.end_time) AS end_time_key,
  
  -- Trip Basics
  t.start_time,
  t.end_time,
  t.trip_status,
  CAST(t.start_time AS DATE) AS trip_date,
  hour(t.start_time) AS trip_hour,
  
  -- Distance & Duration
  t.distance_km,
  t.trip_duration_minutes,
  t.average_speed_kmph,
  t.calculated_avg_speed,
  t.is_speed_mismatch,
  
  -- Fuel Metrics (from fuel_transactions)
  f.fuel_quantity_l,
  f.total_cost AS fuel_cost,
  f.fuel_price_per_l,
  f.fuel_station,
  CASE WHEN f.fuel_quantity_l > 0 THEN t.distance_km / f.fuel_quantity_l ELSE NULL END AS fuel_efficiency_kmpl,
  CASE WHEN t.distance_km > 0 THEN f.total_cost / t.distance_km ELSE NULL END AS fuel_cost_per_km,
  f.is_large_refuel,
  
  -- Performance vs Route
  r.distance_km AS route_distance_km,
  CASE WHEN r.distance_km > 0 THEN (t.distance_km / r.distance_km) * 100 ELSE NULL END AS route_adherence_pct,
  r.expected_avg_speed_kmph,
  CASE WHEN r.expected_avg_speed_kmph > 0 
    THEN ((t.average_speed_kmph - r.expected_avg_speed_kmph) / r.expected_avg_speed_kmph) * 100 
    ELSE NULL 
  END AS speed_variance_pct,
  
  -- Validation Flags
  t.is_completed,
  t.is_cancelled,
  t.is_valid_duration,
  t.is_valid_trip,
  t.is_on_time,
  w.is_adverse_weather
  
FROM workspace.forge.trip_master t
LEFT JOIN workspace.forge.fuel_transactions f ON t.trip_id = f.trip_id
LEFT JOIN workspace.forge.route_master r ON t.route_id = r.route_id
LEFT JOIN workspace.forge.weather w ON t.route_id = w.weather_id
WHERE t.is_valid_trip = TRUE;

In [0]:
%sql
-- Fact: Engine Telemetry (all engine readings)
CREATE OR REPLACE TABLE workspace.insight.fact_engine_telemetry AS
SELECT 
  -- Keys
  engine_record_id,
  vehicle_id,
  trip_id,
  CAST(date_format(timestamp, 'yyyyMMdd') AS INT) AS record_date_key,
  hour(timestamp) * 60 + minute(timestamp) AS record_time_key,
  timestamp,
  
  -- Engine Identifiers
  engine_serial_no,
  engine_model,
  
  -- Core Metrics
  engine_rpm,
  engine_load,
  engine_status,
  engine_health_score,
  
  -- Temperature Monitoring
  coolant_temperature,
  oil_temperature,
  exhaust_temperature,
  is_overheating,
  is_valid_coolant_temp,
  is_valid_oil_temp,
  
  -- Pressure Monitoring
  coolant_pressure,
  oil_pressure,
  fuel_pressure,
  turbo_boost,
  
  -- Electrical
  battery_voltage,
  
  -- Fuel System
  fuel_injection_rate,
  
  -- Vibration & Sensors
  engine_vibration,
  knock_sensor,
  
  -- Diagnostics
  fault_code,
  warning_level,
  has_fault,
  is_critical_warning,
  
  -- Performance Indicators
  is_healthy,
  is_high_load,
  engine_performance,
  custom_health_score,
  health_score_mismatch,
  has_health_score_issue
  
FROM workspace.forge.core_engine;

In [0]:
%sql
-- Fact: Driver Behavior (all behavior readings)
CREATE OR REPLACE TABLE workspace.insight.fact_driver_behavior AS
SELECT 
  -- Keys
  behaviour_id,
  vehicle_id,
  driver_id,
  trip_id,
  CAST(date_format(timestamp, 'yyyyMMdd') AS INT) AS record_date_key,
  hour(timestamp) * 60 + minute(timestamp) AS record_time_key,
  timestamp,
  
  -- Speed Metrics
  average_speed,
  maximum_speed,
  overspeed_events,
  
  -- Harsh Events
  harsh_braking_count,
  rapid_acceleration_count,
  hard_cornering_count,
  lane_departure_events,
  
  -- Idle & Efficiency
  idle_time,
  eco_driving_score,
  
  -- Driving Hours
  continuous_driving_hours,
  night_driving_hours,
  is_excessive_hours,
  is_night_driver,
  
  -- Safety Scores
  driver_fatigue_score,
  driver_distraction_score,
  is_fatigued,
  is_distracted,
  
  -- Safety Compliance
  seatbelt_status,
  phone_usage,
  is_unsafe_conditions,
  
  -- Risk Assessment
  composite_risk_score,
  is_high_risk_driving
  
FROM workspace.forge.driver_behavior;

In [0]:
%sql
-- Fact: Maintenance
CREATE OR REPLACE TABLE workspace.insight.fact_maintenance AS
SELECT 
  -- Keys
  service_id,
  vehicle_id,
  CAST(date_format(service_date, 'yyyyMMdd') AS INT) AS service_date_key,
  
  -- Service Details
  engine_serial_no,
  service_date,
  service_type,
  service_category,
  service_status,
  year(service_date) AS service_year,
  month(service_date) AS service_month,
  hour(service_date) AS service_hour,
  
  -- Diagnosis
  fault_code,
  diagnosis,
  has_fault_code,
  fault_code_count,
  
  -- Service Provider & Location
  loc.location_key AS workshop_location_key,
  m.technician,
  m.workshop,
  
  -- Work Performed
  parts_replaced,
  
  -- Costs
  labour_cost,
  parts_cost,
  total_cost,
  calculated_total_cost,
  is_cost_mismatch,
  cost_difference,
  cost_per_hour,
  is_expensive_repair,
  
  -- Downtime
  downtime_hours,
  downtime_days,
  is_high_downtime,
  
  -- Next Service
  next_service_due,
  days_until_next_service,
  is_overdue,
  
  -- Warranty
  warranty_claim,
  is_warranty_claim,
  
  -- Service Type Flags
  is_preventive,
  is_breakdown
  
FROM workspace.forge.maintenance m
LEFT JOIN workspace.insight.dim_location loc ON m.workshop = loc.city;

In [0]:
%sql
-- Fact: Insurance Claim
CREATE OR REPLACE TABLE workspace.insight.fact_insurance_claim AS
SELECT 
  -- Keys
  claim_id,
  vehicle_id,
  driver_id,
  trip_id,
  CAST(date_format(accident_timestamp, 'yyyyMMdd') AS INT) AS accident_date_key,
  accident_timestamp,
  
  -- Location & Conditions
  loc.location_key AS accident_location_key,
  c.accident_location,
  c.weather_condition,
  c.road_condition,
  is_adverse_weather,
  is_poor_road,
  
  -- Accident Details
  collision_type,
  damaged_component,
  severity,
  is_severe,
  
  -- Financial
  estimated_repair_cost,
  claim_amount,
  cost_difference,
  is_high_value_claim,
  is_claim_over_estimate,
  
  -- Fraud Detection
  fraud_flag,
  is_potential_fraud,
  
  -- Status
  claim_status
  
FROM workspace.forge.insurance_claims c
LEFT JOIN workspace.insight.dim_location loc ON c.accident_location = loc.city;

In [0]:
%sql
-- Fact: Fuel Transaction
CREATE OR REPLACE TABLE workspace.insight.fact_fuel_transaction AS
SELECT 
  -- Keys
  fuel_id,
  vehicle_id,
  trip_id,
  CAST(date_format(timestamp, 'yyyyMMdd') AS INT) AS transaction_date_key,
  timestamp,
  CAST(timestamp AS DATE) AS transaction_date,
  
  -- Location & Station
  loc.location_key AS fuel_station_location_key,
  f.fuel_station,
  f.fuel_type,
  f.payment_method,
  
  -- Quantities & Pricing
  fuel_quantity_l,
  fuel_price_per_l,
  total_cost,
  calculated_total_cost,
  is_cost_mismatch,
  
  -- Validation
  is_valid_quantity,
  is_valid_price,
  is_large_refuel,
  is_complete
  
FROM workspace.forge.fuel_transactions f
LEFT JOIN workspace.insight.dim_location loc ON f.fuel_station = loc.city;

In [0]:
%sql
-- Aggregated Fact: Trip Daily Summary
CREATE OR REPLACE TABLE workspace.insight.fact_trip_daily_summary AS
SELECT 
  -- Keys
  vehicle_id,
  driver_id,
  trip_date_key,
  trip_date,
  
  -- Trip Counts
  COUNT(*) AS total_trips,
  SUM(CASE WHEN is_completed THEN 1 ELSE 0 END) AS completed_trips,
  SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) AS cancelled_trips,
  
  -- Distance & Duration
  SUM(distance_km) AS total_distance_km,
  AVG(distance_km) AS avg_distance_per_trip,
  SUM(trip_duration_minutes) AS total_duration_minutes,
  
  -- Fuel
  SUM(fuel_quantity_l) AS total_fuel_quantity_l,
  SUM(fuel_cost) AS total_fuel_cost,
  AVG(fuel_efficiency_kmpl) AS avg_fuel_efficiency_kmpl,
  
  -- Performance
  AVG(average_speed_kmph) AS avg_speed_kmph,
  MAX(average_speed_kmph) AS max_speed_kmph,
  
  -- Risk Events (will join with behavior)
  0 AS total_harsh_braking,
  0 AS total_rapid_acceleration,
  0 AS total_overspeed_events,
  0.0 AS avg_risk_score,
  
  -- Financial (placeholders)
  SUM(distance_km) * 50 AS total_trip_revenue,
  SUM(fuel_cost) AS total_trip_cost,
  (SUM(distance_km) * 50) - SUM(fuel_cost) AS gross_profit
  
FROM workspace.insight.fact_trip_detail
GROUP BY vehicle_id, driver_id, trip_date_key, trip_date;

In [0]:
%sql
-- Aggregated Fact: Vehicle Monthly Performance
CREATE OR REPLACE TABLE workspace.insight.fact_vehicle_monthly AS
WITH trip_agg AS (
  SELECT 
    vehicle_id,
    date_format(trip_date, 'yyyy-MM') AS year_month,
    year(trip_date) AS year,
    month(trip_date) AS month,
    COUNT(*) AS total_trips,
    SUM(CASE WHEN is_completed THEN 1 ELSE 0 END) AS completed_trips,
    SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) AS cancelled_trips,
    SUM(distance_km) AS total_distance_km,
    AVG(distance_km) AS avg_trip_distance_km,
    SUM(trip_duration_minutes) / 60.0 AS total_trip_hours,
    COUNT(DISTINCT trip_date) AS active_days,
    SUM(fuel_quantity_l) AS total_fuel_quantity_l,
    SUM(fuel_cost) AS total_fuel_cost,
    AVG(fuel_efficiency_kmpl) AS avg_fuel_efficiency_kmpl
  FROM workspace.insight.fact_trip_detail
  GROUP BY vehicle_id, date_format(trip_date, 'yyyy-MM'), year(trip_date), month(trip_date)
),
maint_agg AS (
  SELECT 
    vehicle_id,
    date_format(service_date, 'yyyy-MM') AS year_month,
    COUNT(*) AS maintenance_event_count,
    SUM(CASE WHEN is_preventive THEN 1 ELSE 0 END) AS preventive_count,
    SUM(CASE WHEN is_breakdown THEN 1 ELSE 0 END) AS breakdown_count,
    SUM(total_cost) AS total_maintenance_cost,
    SUM(downtime_hours) AS total_downtime_hours
  FROM workspace.insight.fact_maintenance
  GROUP BY vehicle_id, date_format(service_date, 'yyyy-MM')
),
claim_agg AS (
  SELECT 
    vehicle_id,
    date_format(accident_timestamp, 'yyyy-MM') AS year_month,
    COUNT(*) AS claim_count,
    SUM(claim_amount) AS total_claim_amount,
    SUM(CASE WHEN is_severe THEN 1 ELSE 0 END) AS severe_claim_count
  FROM workspace.insight.fact_insurance_claim
  GROUP BY vehicle_id, date_format(accident_timestamp, 'yyyy-MM')
)
SELECT 
  t.vehicle_id,
  t.year_month,
  t.year,
  t.month,
  
  -- Trip Metrics
  t.total_trips,
  t.completed_trips,
  t.cancelled_trips,
  CASE WHEN t.total_trips > 0 THEN (t.completed_trips * 100.0 / t.total_trips) ELSE 0 END AS trip_completion_rate,
  t.total_distance_km,
  t.avg_trip_distance_km,
  t.total_trip_hours,
  
  -- Utilization
  t.active_days,
  CASE WHEN 30 > 0 THEN (t.active_days * 100.0 / 30) ELSE 0 END AS utilization_rate,
  CASE WHEN t.active_days > 0 THEN (t.total_trips * 1.0 / t.active_days) ELSE 0 END AS trips_per_active_day,
  30 - t.active_days AS idle_days,
  
  -- Fuel Performance
  t.total_fuel_quantity_l,
  t.total_fuel_cost,
  t.avg_fuel_efficiency_kmpl,
  CASE WHEN t.total_distance_km > 0 THEN (t.total_fuel_cost / t.total_distance_km) ELSE 0 END AS fuel_cost_per_km,
  
  -- Maintenance
  COALESCE(m.maintenance_event_count, 0) AS maintenance_event_count,
  COALESCE(m.preventive_count, 0) AS preventive_count,
  COALESCE(m.breakdown_count, 0) AS breakdown_count,
  COALESCE(m.total_maintenance_cost, 0) AS total_maintenance_cost,
  COALESCE(m.total_downtime_hours, 0) AS total_downtime_hours,
  
  -- Insurance
  COALESCE(c.claim_count, 0) AS claim_count,
  COALESCE(c.total_claim_amount, 0) AS total_claim_amount,
  COALESCE(c.severe_claim_count, 0) AS severe_claim_count,
  
  -- Engine Health (placeholders - would aggregate from telemetry)
  0.0 AS avg_engine_health_score,
  0.0 AS min_engine_health_score,
  0 AS critical_warning_count,
  0 AS fault_code_count,
  
  -- Financial
  t.total_distance_km * 50 AS total_revenue,
  t.total_fuel_cost + COALESCE(m.total_maintenance_cost, 0) AS total_cost,
  (t.total_distance_km * 50) - (t.total_fuel_cost + COALESCE(m.total_maintenance_cost, 0)) AS gross_profit,
  CASE WHEN (t.total_distance_km * 50) > 0 
    THEN (((t.total_distance_km * 50) - (t.total_fuel_cost + COALESCE(m.total_maintenance_cost, 0))) / (t.total_distance_km * 50)) * 100 
    ELSE 0 
  END AS profit_margin,
  CASE WHEN t.total_distance_km > 0 THEN ((t.total_fuel_cost + COALESCE(m.total_maintenance_cost, 0)) / t.total_distance_km) ELSE 0 END AS cost_per_km,
  50 AS revenue_per_km,
  
  -- Performance Flags
  FALSE AS is_high_performer,
  FALSE AS is_underutilized,
  FALSE AS needs_attention,
  FALSE AS has_overdue_service
  
FROM trip_agg t
LEFT JOIN maint_agg m ON t.vehicle_id = m.vehicle_id AND t.year_month = m.year_month
LEFT JOIN claim_agg c ON t.vehicle_id = c.vehicle_id AND t.year_month = c.year_month;

In [0]:
%sql
-- Aggregated Fact: Driver Monthly Performance
CREATE OR REPLACE TABLE workspace.insight.fact_driver_monthly AS
WITH trip_agg AS (
  SELECT 
    driver_id,
    date_format(trip_date, 'yyyy-MM') AS year_month,
    year(trip_date) AS year,
    month(trip_date) AS month,
    COUNT(*) AS total_trips,
    SUM(CASE WHEN is_completed THEN 1 ELSE 0 END) AS completed_trips,
    SUM(distance_km) AS total_distance_km,
    SUM(trip_duration_minutes) / 60.0 AS total_driving_hours,
    COUNT(DISTINCT trip_date) AS active_days,
    AVG(distance_km) AS avg_trip_distance_km,
    AVG(fuel_efficiency_kmpl) AS avg_fuel_efficiency_kmpl,
    AVG(average_speed_kmph) AS avg_speed_kmph,
    MAX(average_speed_kmph) AS max_speed_kmph
  FROM workspace.insight.fact_trip_detail
  GROUP BY driver_id, date_format(trip_date, 'yyyy-MM'), year(trip_date), month(trip_date)
),
claim_agg AS (
  SELECT 
    driver_id,
    date_format(accident_timestamp, 'yyyy-MM') AS year_month,
    COUNT(*) AS claim_count,
    SUM(claim_amount) AS total_claim_amount,
    SUM(CASE WHEN is_severe THEN 1 ELSE 0 END) AS severe_claim_count
  FROM workspace.insight.fact_insurance_claim
  GROUP BY driver_id, date_format(accident_timestamp, 'yyyy-MM')
)
SELECT 
  t.driver_id,
  t.year_month,
  t.year,
  t.month,
  
  -- Trip Metrics
  t.total_trips,
  t.completed_trips,
  t.total_distance_km,
  t.total_driving_hours,
  t.active_days,
  CASE WHEN t.active_days > 0 THEN (t.total_trips * 1.0 / t.active_days) ELSE 0 END AS trips_per_day,
  t.avg_trip_distance_km,
  
  -- Time Patterns (placeholders)
  0.0 AS night_driving_hours,
  0 AS night_trip_count,
  0.0 AS night_trip_pct,
  
  -- Safety & Behavior (placeholders - would aggregate from behavior telemetry)
  0.0 AS avg_composite_risk_score,
  0.0 AS max_risk_score,
  0 AS total_harsh_braking,
  0 AS total_rapid_acceleration,
  0 AS total_hard_cornering,
  0 AS total_overspeed_events,
  0 AS total_lane_departure,
  0.0 AS avg_harsh_events_per_trip,
  
  -- Fatigue & Violations (placeholders)
  0.0 AS avg_fatigue_score,
  0.0 AS avg_distraction_score,
  0.0 AS max_continuous_driving_hours,
  0 AS excessive_hours_count,
  0 AS phone_usage_violations,
  0 AS seatbelt_violations,
  
  -- Eco Driving
  0.0 AS avg_eco_driving_score,
  t.avg_fuel_efficiency_kmpl,
  0.0 AS total_idle_time_hours,
  
  -- Insurance
  COALESCE(c.claim_count, 0) AS claim_count,
  0 AS at_fault_claim_count,
  COALESCE(c.total_claim_amount, 0) AS total_claim_amount,
  COALESCE(c.severe_claim_count, 0) AS severe_claim_count,
  
  -- Speed
  t.avg_speed_kmph,
  t.max_speed_kmph,
  0 AS overspeed_trip_count,
  0.0 AS overspeed_trip_pct,
  
  -- Performance Ratings
  'Medium Risk' AS risk_category,
  'B' AS safety_rating,
  'Good' AS performance_rating,
  FALSE AS is_high_risk_driver,
  FALSE AS needs_training,
  FALSE AS is_top_performer,
  FALSE AS requires_intervention
  
FROM trip_agg t
LEFT JOIN claim_agg c ON t.driver_id = c.driver_id AND t.year_month = c.year_month;

In [0]:
%sql
-- Aggregated Fact: Fleet Daily KPIs
CREATE OR REPLACE TABLE workspace.insight.fact_fleet_daily_kpi AS
WITH trip_daily AS (
  SELECT 
    trip_date,
    trip_date_key,
    COUNT(DISTINCT vehicle_id) AS active_vehicles,
    COUNT(DISTINCT driver_id) AS active_drivers,
    COUNT(*) AS total_trips,
    SUM(CASE WHEN is_completed THEN 1 ELSE 0 END) AS completed_trips,
    SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) AS cancelled_trips,
    SUM(distance_km) AS total_distance_km,
    AVG(distance_km) AS avg_distance_per_trip,
    SUM(fuel_quantity_l) AS total_fuel_quantity_l,
    SUM(fuel_cost) AS total_fuel_cost,
    AVG(fuel_efficiency_kmpl) AS avg_fuel_efficiency_kmpl
  FROM workspace.insight.fact_trip_detail
  GROUP BY trip_date, trip_date_key
),
maint_daily AS (
  SELECT 
    CAST(service_date AS DATE) AS service_date,
    COUNT(*) AS maintenance_events,
    SUM(total_cost) AS total_maintenance_cost,
    COUNT(DISTINCT vehicle_id) AS vehicles_in_service,
    SUM(downtime_hours) AS total_downtime_hours
  FROM workspace.insight.fact_maintenance
  GROUP BY CAST(service_date AS DATE)
),
claim_daily AS (
  SELECT 
    CAST(accident_timestamp AS DATE) AS accident_date,
    COUNT(*) AS claim_count,
    SUM(CASE WHEN is_severe THEN 1 ELSE 0 END) AS severe_claim_count,
    SUM(claim_amount) AS total_claim_amount
  FROM workspace.insight.fact_insurance_claim
  GROUP BY CAST(accident_timestamp AS DATE)
)
SELECT 
  t.trip_date_key AS kpi_date_key,
  t.trip_date AS kpi_date,
  
  -- Fleet Size (from dimension - using placeholders)
  500 AS total_vehicles,
  t.active_vehicles,
  COALESCE(m.vehicles_in_service, 0) AS vehicles_in_maintenance,
  700 AS total_drivers,
  t.active_drivers,
  
  -- Operations
  t.total_trips,
  t.completed_trips,
  t.cancelled_trips,
  CASE WHEN t.total_trips > 0 THEN (t.completed_trips * 100.0 / t.total_trips) ELSE 0 END AS trip_completion_rate,
  t.total_distance_km,
  t.avg_distance_per_trip,
  
  -- Utilization
  CASE WHEN 500 > 0 THEN (t.active_vehicles * 100.0 / 500) ELSE 0 END AS fleet_utilization_rate,
  CASE WHEN 700 > 0 THEN (t.active_drivers * 100.0 / 700) ELSE 0 END AS driver_utilization_rate,
  CASE WHEN t.active_vehicles > 0 THEN (t.total_trips * 1.0 / t.active_vehicles) ELSE 0 END AS avg_trips_per_vehicle,
  CASE WHEN t.active_drivers > 0 THEN (t.total_trips * 1.0 / t.active_drivers) ELSE 0 END AS avg_trips_per_driver,
  
  -- Fuel
  t.total_fuel_quantity_l,
  t.total_fuel_cost,
  t.avg_fuel_efficiency_kmpl,
  CASE WHEN t.total_distance_km > 0 THEN (t.total_fuel_cost / t.total_distance_km) ELSE 0 END AS fuel_cost_per_km,
  
  -- Maintenance
  COALESCE(m.maintenance_events, 0) AS maintenance_events,
  COALESCE(m.total_maintenance_cost, 0) AS total_maintenance_cost,
  COALESCE(m.vehicles_in_service, 0) AS vehicles_in_service,
  COALESCE(m.total_downtime_hours, 0) AS total_downtime_hours,
  
  -- Safety
  COALESCE(c.claim_count, 0) AS claim_count,
  COALESCE(c.severe_claim_count, 0) AS severe_claim_count,
  COALESCE(c.total_claim_amount, 0) AS total_claim_amount,
  0 AS total_harsh_events,
  0 AS high_risk_trip_count,
  
  -- Engine Health (placeholders)
  0.0 AS avg_engine_health_score,
  0 AS critical_warning_count,
  0 AS vehicles_with_faults,
  
  -- Financial
  t.total_distance_km * 50 AS total_revenue,
  t.total_fuel_cost + COALESCE(m.total_maintenance_cost, 0) AS total_operating_cost,
  (t.total_distance_km * 50) - (t.total_fuel_cost + COALESCE(m.total_maintenance_cost, 0)) AS gross_profit,
  CASE WHEN (t.total_distance_km * 50) > 0 
    THEN (((t.total_distance_km * 50) - (t.total_fuel_cost + COALESCE(m.total_maintenance_cost, 0))) / (t.total_distance_km * 50)) * 100 
    ELSE 0 
  END AS profit_margin,
  CASE WHEN t.total_distance_km > 0 THEN ((t.total_fuel_cost + COALESCE(m.total_maintenance_cost, 0)) / t.total_distance_km) ELSE 0 END AS cost_per_km,
  50 AS revenue_per_km,
  
  -- Performance vs Target (placeholders)
  100.0 AS trips_vs_target_pct,
  100.0 AS revenue_vs_target_pct,
  100.0 AS cost_vs_target_pct
  
FROM trip_daily t
LEFT JOIN maint_daily m ON t.trip_date = m.service_date
LEFT JOIN claim_daily c ON t.trip_date = c.accident_date;